# LSTM Training - Prediksi Penjualan UMKM Teras Rasa

Notebook ini berisi proses:
1. Load & Preprocessing data (stroberi, mapping tanggal, penyesuaian jumlah hari)
2. Grid Search tuning hyperparameter LSTM per menu
3. Training 6 model LSTM (satu per menu)
4. Evaluasi & penyimpanan model + metadata

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import warnings
from calendar import monthrange
from itertools import product

import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

print('TensorFlow version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 1. Load Data dari Excel

In [ ]:
EXCEL_PATH = 'DataPenjualanUMKMTerasRasa.xlsx'
MENUS = ['mie ayam', 'alpukat', 'mangga', 'jeruk', 'jambu', 'strobery']

xl = pd.ExcelFile(EXCEL_PATH)
print('Sheet:', xl.sheet_names)

all_data = []
for sheet in xl.sheet_names:
    df = pd.read_excel(xl, sheet_name=sheet)
    print(f'  {sheet}: {len(df)} baris')
    all_data.append(df)

raw = pd.concat(all_data, ignore_index=True).reset_index(drop=True)
print(f'\nTotal: {len(raw)} baris, kolom: {raw.columns.tolist()}')
raw.head()

## 2. Preprocessing Stroberi

- Hari tutup (semua kolom == 0): biarkan 0 semua
- Hari non-tutup yang stroberi == 0: isi dengan rata-rata (alpukat, mangga, jeruk, jambu)

In [ ]:
filled_count = 0
for idx, row in raw.iterrows():
    is_buka = row[MENUS].sum() > 0
    if is_buka and row['strobery'] == 0:
        avg_val = row[['alpukat', 'mangga', 'jeruk', 'jambu']].mean()
        raw.at[idx, 'strobery'] = round(avg_val)
        filled_count += 1

print(f'Stroberi diisi (hari non-tutup): {filled_count} baris')
print(f'Strobery == 0 tersisa (hari tutup): {(raw["strobery"] == 0).sum()} baris')

## 3. Mapping Tanggal & Penyesuaian Jumlah Hari

| Original | Target | Hari Original | Hari Target |
|----------|--------|---------------|-------------|
| Nov 2021 | Jan 2026 | 30 | 31 |
| Des 2021 | Feb 2026 | 31 | 28 |
| Jan 2022 | Mar 2026 | 31 | 31 |
| Feb 2022 | Apr 2026 | 28 | 30 |
| Mar 2022 | Mei 2026 | 31 | 31 |
| Apr 2022 | Jun 2026 | 30 | 30 |

- Kekurangan: sintesis dengan interpolasi linear
- Kelebihan: hapus dari akhir

In [ ]:
MONTH_MAP = {11:(2026,1), 12:(2026,2), 1:(2026,3), 2:(2026,4), 3:(2026,5), 4:(2026,6)}
TARGET_DAYS = {1:31, 2:28, 3:31, 4:30, 5:31, 6:30}

def map_date_safe(d):
    ny, nm = MONTH_MAP[d.month]
    max_day = monthrange(ny, nm)[1]
    return pd.Timestamp(year=ny, month=nm, day=min(d.day, max_day))

raw['date'] = raw['date'].apply(map_date_safe)

result_frames = []
for nm, target_n in TARGET_DAYS.items():
    month_data = raw[raw['date'].dt.month == nm].copy().sort_values('date').reset_index(drop=True)
    current_n = len(month_data)
    
    if current_n == target_n:
        result_frames.append(month_data)
    elif current_n > target_n:
        print(f'  Bulan {nm:02d}: hapus {current_n - target_n} baris dari akhir')
        result_frames.append(month_data.head(target_n))
    else:
        old_idx = np.linspace(0, target_n - 1, current_n)
        new_idx = np.arange(target_n)
        synthesized = pd.DataFrame()
        synthesized['date'] = pd.date_range(start=f'2026-{nm:02d}-01', periods=target_n)
        for col in MENUS:
            vals = month_data[col].values.astype(float)
            new_vals = np.interp(new_idx, old_idx, vals)
            synthesized[col] = np.round(new_vals).astype(int)
        print(f'  Bulan {nm:02d}: sintesis {target_n - current_n} baris')
        result_frames.append(synthesized)

df = pd.concat(result_frames, ignore_index=True)
print(f'\nTotal baris akhir: {len(df)}')
print(df.groupby(df['date'].dt.to_period('M')).size())

## 4. Simpan Data Bersih

In [ ]:
df.to_csv('data_clean.csv', index=False)
print('Tersimpan: data_clean.csv')
df.head(10)

## 5. Explorasi Data

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
for i, menu in enumerate(MENUS):
    ax = axes[i // 2, i % 2]
    ax.plot(df['date'], df[menu], linewidth=0.8)
    ax.set_title(menu.title())
    ax.set_ylabel('Porsi')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Penjualan Harian per Menu (Jan-Jun 2026)', fontsize=14)
plt.tight_layout()
plt.savefig('plot_data_clean.png', dpi=100)
plt.show()

## 6. Fungsi LSTM

In [ ]:
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length, 0])
        y.append(data[i+seq_length, 0])
    return np.array(X), np.array(y)


def build_lstm_model(seq_length, lstm_units, learning_rate):
    model = tf.keras.Sequential([
        tf.keras.layers.LSTM(lstm_units, input_shape=(seq_length, 1), return_sequences=True),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.LSTM(lstm_units // 2),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1)
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae']
    )
    return model


def evaluate_predictions(actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    nonzero_mask = actual != 0
    if nonzero_mask.sum() > 0:
        mape = np.mean(np.abs((actual[nonzero_mask] - predicted[nonzero_mask]) / actual[nonzero_mask])) * 100
    else:
        mape = 0.0
    return {'MAE': round(mae, 4), 'RMSE': round(rmse, 4), 'MAPE': round(mape, 2)}

## 7. Grid Search Hyperparameter Tuning

Parameter yang dituning:
- `seq_length`: [7, 14, 30]
- `lstm_units`: [32, 64, 128]
- `learning_rate`: [0.001, 0.0005]

Total: 18 kombinasi per menu. Best model berdasarkan MAE terendah.

In [ ]:
PARAM_GRID = {
    'seq_length': [7, 14, 30],
    'lstm_units': [32, 64, 128],
    'learning_rate': [0.001, 0.0005]
}

EPOCHS = 100
BATCH_SIZE = 8
TEST_RATIO = 0.2

all_combos = list(product(
    PARAM_GRID['seq_length'],
    PARAM_GRID['lstm_units'],
    PARAM_GRID['learning_rate']
))
print(f'Total kombinasi per menu: {len(all_combos)}')

## 8. Training 6 Model + Grid Search

In [ ]:
MODEL_DIR = '../app/ml_models'
os.makedirs(MODEL_DIR, exist_ok=True)

results = {}
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, monitor='val_loss'),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-6)
]

for menu in MENUS:
    print(f'\n{"="*60}')
    print(f'MENU: {menu.upper()}')
    print(f'{"="*60}')
    
    raw_data = df[menu].values.reshape(-1, 1).astype(float)
    
    best_mae = float('inf')
    best_config = {}
    best_model = None
    best_scaler = None
    best_history = None
    grid_results = []
    
    for seq_len, units, lr in all_combos:
        # Scale data
        scaler = MinMaxScaler()
        scaled = scaler.fit_transform(raw_data)
        
        # Create sequences
        X, y = create_sequences(scaled, seq_len)
        if len(X) < 20:
            continue
        
        X = X.reshape(X.shape[0], X.shape[1], 1)
        
        # Split
        split = int(len(X) * (1 - TEST_RATIO))
        X_train, X_test = X[:split], X[split:]
        y_train, y_test = y[:split], y[split:]
        
        # Build & train
        tf.keras.backend.clear_session()
        model = build_lstm_model(seq_len, units, lr)
        history = model.fit(
            X_train, y_train,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_split=0.1,
            callbacks=callbacks,
            verbose=0
        )
        
        # Evaluate
        pred_scaled = model.predict(X_test, verbose=0)
        pred = scaler.inverse_transform(pred_scaled).flatten()
        actual = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
        
        metrics = evaluate_predictions(actual, pred)
        grid_results.append({
            'seq_length': seq_len, 'lstm_units': units, 'learning_rate': lr,
            'epochs_ran': len(history.history['loss']), **metrics
        })
        
        if metrics['MAE'] < best_mae:
            best_mae = metrics['MAE']
            best_config = {'seq_length': seq_len, 'lstm_units': units, 'learning_rate': lr}
            best_model = model
            best_scaler = scaler
            best_history = history
        
        print(f'  seq={seq_len}, units={units}, lr={lr} → MAE={metrics["MAE"]:.4f}, RMSE={metrics["RMSE"]:.4f}, MAPE={metrics["MAPE"]:.2f}%')
    
    # Simpan best model & scaler
    model_path = os.path.join(MODEL_DIR, f'{menu.replace(" ", "_")}_lstm_model.h5')
    scaler_path = f'{menu.replace(" ", "_")}_scaler.pkl'
    best_model.save(model_path)
    joblib.dump(best_scaler, scaler_path)
    
    # Retrain best model di full data untuk deployment
    tf.keras.backend.clear_session()
    full_scaler = MinMaxScaler()
    full_scaled = full_scaler.fit_transform(raw_data)
    X_full, y_full = create_sequences(full_scaled, best_config['seq_length'])
    X_full = X_full.reshape(X_full.shape[0], X_full.shape[1], 1)
    
    final_model = build_lstm_model(best_config['seq_length'], best_config['lstm_units'], best_config['learning_rate'])
    final_model.fit(X_full, y_full, epochs=50, batch_size=BATCH_SIZE, verbose=0)
    final_model.save(model_path)
    joblib.dump(full_scaler, scaler_path)
    
    best_grid = min(grid_results, key=lambda x: x['MAE'])
    results[menu] = {
        'best_params': best_config,
        'best_metrics': best_grid,
        'all_grid_results': grid_results
    }
    
    print(f'\n  ✅ Best: seq={best_config["seq_length"]}, units={best_config["lstm_units"]}, lr={best_config["learning_rate"]}')
    print(f'     MAE={best_grid["MAE"]:.4f}, RMSE={best_grid["RMSE"]:.4f}, MAPE={best_grid["MAPE"]:.2f}%')
    print(f'     Model: {model_path}')
    print(f'     Scaler: {scaler_path}')

## 9. Ringkasan Evaluasi

In [ ]:
summary = []
for menu in MENUS:
    r = results[menu]
    summary.append({
        'Menu': menu.title(),
        'Seq Length': r['best_params']['seq_length'],
        'LSTM Units': r['best_params']['lstm_units'],
        'LR': r['best_params']['learning_rate'],
        'MAE': r['best_metrics']['MAE'],
        'RMSE': r['best_metrics']['RMSE'],
        'MAPE (%)': r['best_metrics']['MAPE']
    })

summary_df = pd.DataFrame(summary)
print('=== RINGKASAN BEST MODEL PER MENU ===')
print(summary_df.to_string(index=False))
summary_df.to_csv('evaluation_summary.csv', index=False)

## 10. Plot Training Loss & Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

for i, menu in enumerate(MENUS):
    ax = axes[i // 2, i % 2]
    r = results[menu]
    
    raw_data = df[menu].values.reshape(-1, 1).astype(float)
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(raw_data)
    seq_len = r['best_params']['seq_length']
    X_all, y_all = create_sequences(scaled, seq_len)
    X_all = X_all.reshape(X_all.shape[0], X_all.shape[1], 1)
    
    model = tf.keras.models.load_model(f'../app/ml_models/{menu.replace(" ", "_")}_lstm_model.h5')
    pred_scaled = model.predict(X_all, verbose=0)
    pred = scaler.inverse_transform(pred_scaled).flatten()
    actual = scaler.inverse_transform(y_all.reshape(-1, 1)).flatten()
    
    dates_plot = df['date'].values[seq_len:]
    ax.plot(dates_plot, actual, label='Actual', linewidth=1, alpha=0.8)
    ax.plot(dates_plot, pred, label='Predicted', linewidth=1, alpha=0.8, linestyle='--')
    ax.set_title(f'{menu.title()} (MAE={r["best_metrics"]["MAE"]:.2f})')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Actual vs Predicted per Menu', fontsize=14)
plt.tight_layout()
plt.savefig('plot_actual_vs_predicted.png', dpi=100)
plt.show()

## 11. Simpan Metadata JSON

In [ ]:
from datetime import datetime

metadata = {
    'trained_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'data_range': {
        'start': str(df['date'].min().date()),
        'end': str(df['date'].max().date()),
        'total_days': len(df)
    },
    'menus': {}
}

for menu in MENUS:
    r = results[menu]
    metadata['menus'][menu] = {
        'model_file': f'{menu.replace(" ", "_")}_lstm_model.h5',
        'scaler_file': f'{menu.replace(" ", "_")}_scaler.pkl',
        'best_params': r['best_params'],
        'metrics': {k: v for k, v in r['best_metrics'].items()}
    }

with open('metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Tersimpan: metadata.json')
print(json.dumps(metadata, indent=2))

## 12. Daftar File yang Dihasilkan

| File | Lokasi | Keterangan |
|------|--------|------------|
| `{menu}_lstm_model.h5` | `app/ml_models/` | Model LSTM per menu |
| `{menu}_scaler.pkl` | `model/` | Scaler MinMax per menu |
| `data_clean.csv` | `model/` | Data bersih setelah preprocessing |
| `metadata.json` | `model/` | Best params & metrics |
| `evaluation_summary.csv` | `model/` | Tabel ringkasan evaluasi |